In [2]:
import sys
print(sys.executable)

/home/ahmad/movie_recom/myenv/bin/python


In [3]:
import pandas as pd 

In [4]:
df = pd.read_csv("./data/movies_metadata.csv")

# df.head()

features_to_keep = [
    "id",
    "title",
    "overview",  # Bring this back
    "genres",
    "belongs_to_collection",
    "vote_average",
    "vote_count",
    "popularity",
    "imdb_id"
]

usefull_data = df[features_to_keep]

usefull_data.head()

/tmp/ipykernel_12192/3624165036.py:1: DtypeWarning: Columns (10: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("./data/movies_metadata.csv")


,id,title,overview,genres,belongs_to_collection,vote_average,vote_count,popularity,imdb_id
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","{'id': 10194, 'name': 'Toy Story Collection', ...",7.7,5415.0,21.946943,tt0114709
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,6.9,2413.0,17.015539,tt0113497
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","{'id': 119050, 'name': 'Grumpy Old Men Collect...",6.5,92.0,11.7129,tt0113228
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,6.1,34.0,3.859495,tt0114885
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]","{'id': 96871, 'name': 'Father of the Bride Col...",5.7,173.0,8.387519,tt0113041


In [5]:
usefull_data.describe()

,vote_average,vote_count
count,45460.000000,45460.000000
mean,5.618207,109.897338
std,1.924216,491.310374
min,0.000000,0.000000
25%,5.000000,3.000000
50%,6.000000,10.000000
75%,6.800000,34.000000
max,10.000000,14075.000000


In [6]:
import pandas as pd

movies = pd.read_csv("./data/movies_metadata.csv", low_memory=False)  #
# credits = pd.read_csv("./data/credits.csv")  #
keywords = pd.read_csv("./data/keywords.csv")  #
links_small = pd.read_csv("./data/links_small.csv")  #[cite: 2]



In [7]:
# STEP 2: ENFORCE INTEGER TYPES & CLEAN IDs
# ==========================================
# Clean Movies 'id'
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)

# Clean Keywords 'id'
keywords['id'] = pd.to_numeric(keywords['id'], errors='coerce')
keywords = keywords.dropna(subset=['id'])
keywords['id'] = keywords['id'].astype(int)

# ==========================================
# STEP 3: MERGE MOVIES AND KEYWORDS
# ==========================================
# We merge on 'id' so each movie gets its keywords
df = movies.merge(keywords, on='id', how='inner')


In [8]:
import ast
# ==========================================
# STEP 4: SELECT ONLY YOUR CHOSEN FEATURES
# ==========================================
columns_to_keep = [
    'id', 
    'title', 
    'genres', 
    'keywords',     
    'overview',     
    'vote_average', 
    'vote_count',   
    'release_date', 
    'popularity',   
]

df_clean = df[columns_to_keep].copy()

# ==========================================
# STEP 5: CLEAN THE COLUMNS
# ==========================================
# Clean 'genres' 
df_clean['genres'] = df_clean['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
df_clean['genre_names'] = df_clean['genres'].apply(lambda x: [i['name'] for i in x])

# Clean 'keywords'
df_clean['keywords'] = df_clean['keywords'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
df_clean['keyword_names'] = df_clean['keywords'].apply(lambda x: [i['name'] for i in x])

# Clean 'overview'
df_clean['overview'] = df_clean['overview'].fillna('')

# Clean 'release_date' and extract 'year'
df_clean['release_date'] = pd.to_datetime(df_clean['release_date'], errors='coerce')
df_clean['year'] = df_clean['release_date'].dt.year.fillna(0).astype(int)

# Ensure vote_count and vote_average are numeric (needed for filtering)
df_clean['vote_count'] = pd.to_numeric(df_clean['vote_count'], errors='coerce').fillna(0)
df_clean['vote_average'] = pd.to_numeric(df_clean['vote_average'], errors='coerce').fillna(0)

df_clean.head()

,id,title,genres,keywords,overview,vote_average,vote_count,release_date,popularity,genre_names,keyword_names,year
0,862,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","Led by Woody, Andy's toys live happily in his ...",7.7,5415.0,1995-10-30,21.946943,"[Animation, Comedy, Family]","[jealousy, toy, boy, friendship, friends, riva...",1995
1,8844,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'id': 10090, 'name': 'board game'}, {'id': 1...",When siblings Judy and Peter discover an encha...,6.9,2413.0,1995-12-15,17.015539,"[Adventure, Fantasy, Family]","[board game, disappearance, based on children'...",1995
2,15602,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392...",A family wedding reignites the ancient feud be...,6.5,92.0,1995-12-22,11.7129,"[Romance, Comedy]","[fishing, best friend, duringcreditsstinger, o...",1995
3,31357,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","[{'id': 818, 'name': 'based on novel'}, {'id':...","Cheated on, mistreated and stepped on, the wom...",6.1,34.0,1995-12-22,3.859495,"[Comedy, Drama, Romance]","[based on novel, interracial relationship, sin...",1995
4,11862,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]","[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...",Just when George Banks has recovered from his ...,5.7,173.0,1995-02-10,8.387519,[Comedy],"[baby, midlife crisis, confidence, aging, daug...",1995


In [9]:
# ==========================================
# STEP 6: CREATE THE "METADATA SOUP"
# ==========================================
def create_soup(x):
    keywords = ' '.join(x['keyword_names']) if isinstance(x['keyword_names'], list) else ''
    genres = ' '.join(x['genre_names']) if isinstance(x['genre_names'], list) else ''
    return f"{keywords} {genres} {x['overview']}"

df_clean['soup'] = df_clean.apply(create_soup, axis=1)

# ==========================================
# STEP 7: FILTER BY VOTE COUNT (Obscure movies)
# ==========================================
# Keep only movies with 50+ votes
df_clean = df_clean[df_clean['vote_count'] >= 50]

# ==========================================
# STEP 7.5: FILTER BY YEAR (2005 ONLY)  <-- NEW STEP
# ==========================================
# Keep only movies released in > 2005
df_clean = df_clean[df_clean['year'] >= 2005]

# ==========================================
# STEP 7.8: RESET THE INDEX (Crucial after filtering)
# ==========================================
df_clean = df_clean.reset_index(drop=True)

# ==========================================
# STEP 8: VERIFY THE FINAL RESULT
# ==========================================
print(f"Final dataset shape: {df_clean.shape}")
print(f"Oldest year: {df_clean['year'].min()}") # Should print 2005
print(f"Newest year: {df_clean['year'].max()}") # Should print 2026 or whatever the latest year in the data is
print("\nSample of the cleaned data:")
print(df_clean[['title', 'vote_average', 'year', 'soup']].head())

Final dataset shape: (5010, 13)
Oldest year: 2005
Newest year: 2020

Sample of the cleaned data:
                title  vote_average  year  \
0  A Sound of Thunder           4.8  2005   
1          The Jacket           6.8  2005   
2            Millions           6.3  2005   
3     The Interpreter           6.2  2005   
4        The Ring Two           5.5  2005   

                                                soup  
0  dying and death time travel romance dinosaur T...  
1  smoking medicine psychology hallucination huma...  
2  female nudity robbery beheading miracle halluc...  
3  new york dictator africa destruction of a civi...  
4  nun based on novel bath tub nightmare son sequ...  


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the vectorizer
# stop_words='english' removes useless words like 'the', 'and', 'is'
tfidf = TfidfVectorizer(stop_words='english')

# Transform the 'soup' column into a matrix of numbers
tfidf_matrix = tfidf.fit_transform(df_clean['soup'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
# Output will be: (Number of movies, Number of unique words)

TF-IDF Matrix Shape: (5010, 22451)


In [11]:
df_clean.head()

,id,title,genres,keywords,overview,vote_average,vote_count,release_date,popularity,genre_names,keyword_names,year,soup
0,10077,A Sound of Thunder,"[{'id': 53, 'name': 'Thriller'}, {'id': 878, '...","[{'id': 3737, 'name': 'dying and death'}, {'id...",When a hunter sent back to the prehistoric era...,4.8,111.0,2005-05-15,4.980191,"[Thriller, Science Fiction, Adventure, Action]","[dying and death, time travel, romance, dinosaur]",2005,dying and death time travel romance dinosaur T...
1,9667,The Jacket,"[{'id': 18, 'name': 'Drama'}, {'id': 9648, 'na...","[{'id': 919, 'name': 'smoking'}, {'id': 1279, ...",A military veteran goes on a journey into the ...,6.8,459.0,2005-03-04,12.092096,"[Drama, Mystery, Thriller, Fantasy]","[smoking, medicine, psychology, hallucination,...",2005,smoking medicine psychology hallucination huma...
2,13373,Millions,"[{'id': 35, 'name': 'Comedy'}, {'id': 80, 'nam...","[{'id': 293, 'name': 'female nudity'}, {'id': ...","Two boys, still grieving the death of their mo...",6.3,106.0,2005-04-08,4.66886,"[Comedy, Crime, Drama]","[female nudity, robbery, beheading, miracle, h...",2005,female nudity robbery beheading miracle halluc...
3,179,The Interpreter,"[{'id': 80, 'name': 'Crime'}, {'id': 53, 'name...","[{'id': 242, 'name': 'new york'}, {'id': 407, ...","After Silvia Broome, an interpreter at United ...",6.2,400.0,2005-04-08,12.047671,"[Crime, Thriller]","[new york, dictator, africa, destruction of a ...",2005,new york dictator africa destruction of a civi...
4,10320,The Ring Two,"[{'id': 18, 'name': 'Drama'}, {'id': 27, 'name...","[{'id': 745, 'name': 'nun'}, {'id': 818, 'name...",Rachel Keller must prevent evil Samara from ta...,5.5,646.0,2005-03-17,16.15818,"[Drama, Horror, Thriller]","[nun, based on novel, bath tub, nightmare, son...",2005,nun based on novel bath tub nightmare son sequ...


In [12]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# 1. Initialize the Tokenizer (Keep the top 10,000 most important words)
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df_clean['soup'])

# 2. Convert text to sequences of integers
sequences = tokenizer.texts_to_sequences(df_clean['soup'])

# 3. Pad the sequences so they are all the same length (e.g., 100 words max)
max_length = 100
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

print(f"Shape of padded sequences: {padded_sequences.shape}")
# Output: (Number of movies, 100)

I0000 00:00:1791094618.771744   12192 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791094618.840371   12192 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791094620.225668   12192 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791094629.703717   12192 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

Shape of padded sequences: (5010, 100)


In [13]:
from tensorflow.keras import layers, models
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Create the Bag-of-Words target (this is what the network will try to predict)
# Convert padded_sequences into a binary matrix of shape (num_movies, vocab_size)
num_movies = padded_sequences.shape[0]
bow_targets = np.zeros((num_movies, vocab_size), dtype=np.float32)

for i, seq in enumerate(padded_sequences):
    # Get unique word IDs in this movie (ignore padding zeros)
    unique_words = set(seq) - {0}
    for word_id in unique_words:
        if word_id < vocab_size:
            bow_targets[i, word_id] = 1.0

print(f"BoW target shape: {bow_targets.shape}")
# Output: (Number of movies, 10000)

# 2. Build the Autoencoder
input_layer = layers.Input(shape=(max_length,), name='input_sequence')
embedding_layer = layers.Embedding(input_dim=vocab_size, output_dim=64)(input_layer)
pooled = layers.GlobalAveragePooling1D()(embedding_layer)
movie_vector_layer = layers.Dense(64, activation='relu', name='movie_embedding')(pooled)
output_layer = layers.Dense(vocab_size, activation='sigmoid', name='bow_output')(movie_vector_layer)

autoencoder = models.Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Build the Encoder (what we'll use to get the vectors)
encoder = models.Model(inputs=input_layer, outputs=movie_vector_layer)

autoencoder.summary()

BoW target shape: (5010, 10000)


E0000 00:00:1791094639.946775   12192 cuda_executor.cc:1690] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
=== Source Location Trace: === 
external/xla/xla/stream_executor/cuda/cuda_status.cc:68

W0000 00:00:1791094640.113820   12446 cuda_executor.cc:1710] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
=== Source Location Trace: === 
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:77
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:107
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:112

W0000 00:00:1791094640.215135   12192 dlopen_checker.cc:38] Could not load dynamic library 'libcudart.so.12'; dlerror: libcudart.so.12: cannot open shared object file: No such file or directory
W0000 00:00:1791094640.215222   12

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_sequence (InputLayer)     │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ movie_embedding (Dense)         │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bow_output (Dense)              │ (None, 10000)          │       650,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,294,160 (4.94 MB)

 Trainable params: 1,294,160 (4.94 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
print("Training the Neural Network...")
autoencoder.fit(
    padded_sequences,   # Input: sequence of 100 word-IDs
    bow_targets,        # Target: bag-of-words of 10000 words
    epochs=20,
    batch_size=256,
    validation_split=0.1
)

Training the Neural Network...
Epoch 1/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 9s 261ms/step - accuracy: 0.3331 - loss: 0.6525 - val_accuracy: 0.8184 - val_loss: 0.4616
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 152ms/step - accuracy: 0.3602 - loss: 0.2442 - val_accuracy: 0.0419 - val_loss: 0.0619
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 164ms/step - accuracy: 0.0809 - loss: 0.0624 - val_accuracy: 0.1138 - val_loss: 0.0362
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 150ms/step - accuracy: 0.0739 - loss: 0.0405 - val_accuracy: 0.1138 - val_loss: 0.0310
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 154ms/step - accuracy: 0.0435 - loss: 0.0344 - val_accuracy: 0.0419 - val_loss: 0.0286
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 151ms/step - accuracy: 0.0368 - loss: 0.0317 - val_accuracy: 0.0419 - val_loss: 0.0271
Epoch 7/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 149ms/step - accuracy: 0.0368 - loss: 0.0300 - val_accuracy: 0.0419 - val_loss: 0.0259
Epoch 8/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 150ms/step - accuracy: 0.0368 - 

In [15]:
movie_vectors = encoder.predict(padded_sequences)
print(f"Movie Vectors shape: {movie_vectors.shape}")
# Output: (Number of movies, 64)

157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
Movie Vectors shape: (5010, 64)


In [16]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# 1. Calculate similarity using the Neural Network vectors
nn_cosine_sim = cosine_similarity(movie_vectors, movie_vectors)

# 2. Create a title-to-index map
indices = pd.Series(df_clean.index, index=df_clean['title']).drop_duplicates()

# 3. Build the recommendation function
def get_nn_recommendations(title, cosine_sim=nn_cosine_sim):
    if title not in indices:
        return f"Movie '{title}' not found."
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11] # Get top 10, skipping the movie itself
    
    movie_indices = [i[0] for i in sim_scores]
    return df_clean[['title', 'year', 'vote_average']].iloc[movie_indices]

# 4. TEST IT!
get_nn_recommendations('Batman Begins')

,title,year,vote_average
1860,A Very Harold & Kumar Christmas,2011,6.0
178,Transamerica,2005,6.9
2173,What to Expect When You're Expecting,2012,5.8
2513,Resolution,2012,6.3
1918,The Thing,2011,6.0
339,The Wind That Shakes the Barley,2006,7.1
4877,Power Rangers,2017,6.3
3917,Afro Samurai,2007,7.3
2565,The Kings of Summer,2013,7.0
216,Running Scared,2006,7.0


In [17]:
def get_smart_recommendations(title, cosine_sim=nn_cosine_sim):
    if title not in indices:
        return f"Movie '{title}' not found."
    
    idx = indices[title]
    
    # Get the pairwise similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the top 50 most similar movies (we get more so we can filter them down)
    sim_scores = sim_scores[1:51]
    movie_indices = [i[0] for i in sim_scores]
    
    # Create a DataFrame of the recommendations using your CLEAN DATA
    recommendations = df_clean[['title', 'year', 'vote_average', 'vote_count']].iloc[movie_indices]
    
    # NOW USE YOUR CLEAN DATA FEATURES TO FILTER!
    # Example 1: Only recommend movies with a rating of 7.0 or higher
    recommendations = recommendations[recommendations['vote_average'] >= 7.0]
    
    # Example 2: Only recommend movies released in the same decade (e.g., 2000s)
    target_year = df_clean.iloc[idx]['year']
    recommendations = recommendations[
        (recommendations['year'] >= target_year - 5) & 
        (recommendations['year'] <= target_year + 5)
    ]
    
    # Return the top 10 after filtering
    return recommendations.head(10)

In [18]:
# Test with a blockbuster movie
get_smart_recommendations('Toy Story')

"Movie 'Toy Story' not found."

In [19]:
# Try a comedy (assuming it's in your 2005+ dataset)
print("--- COMEDY TEST ---")
print(get_smart_recommendations('The Hangover'))

# Try a Sci-Fi movie
print("\n--- SCI-FI TEST ---")
print(get_smart_recommendations('Inception'))

--- COMEDY TEST ---
                                title  year  vote_average  vote_count
1426         How to Train Your Dragon  2010           7.5      4319.0
2391                  Rurouni Kenshin  2012           7.2       174.0
2605                 What's in a Name  2012           7.2       433.0
2376                   The Impossible  2012           7.0      2071.0
2002  The Girl with the Dragon Tattoo  2011           7.2      2479.0

--- SCI-FI TEST ---
                         title  year  vote_average  vote_count
4402   They Call Me Jeeg Robot  2015           7.5       575.0
1065                Departures  2008           7.7       202.0
1160  The Stoning of Soraya M.  2008           8.0        63.0
1949                    Tomboy  2011           7.2       181.0
364               The Departed  2006           7.9      4455.0
1143                        Up  2009           7.8      7048.0
1631               The Fighter  2010           7.2      1512.0
2763                       Ida  201